# Retrieval v5 — một luồng chung HS/Legal

Dev: 752 HS + 520 Legal. Test: 188 HS + 130 Legal. Chọn bằng Dev, khóa cấu hình rồi đánh giá Test qua Auto. Test đã được xem là hồi quy. Không dùng latency làm điều kiện chọn. Nhận diện ngữ nghĩa chỉ là metadata; không định tuyến hoặc lọc nguồn.


In [ ]:
from pathlib import Path
import sys
import subprocess

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from rag.data import read, digest
from rag.db import connect

RUN = ROOT / "artifacts/experiments/run_unified_accuracy_v5"
RESUME_BENCHMARK = False
RUN_RUNTIME_DEMO = False


## 1. Thực nghiệm đã khóa

`RESUME_BENCHMARK=False` chỉ đọc kết quả đã có. Khi bật, runner resume protocol hiện có, sàng lọc M3/Qwen/Gemma, so sánh cascade trên Dev rồi kích hoạt policy và chấm Test. Không chạy đồng thời với runner đang hoạt động. Source/data đã đổi cần một protocol/lượt thực nghiệm mới.


In [ ]:
if RESUME_BENCHMARK:
    subprocess.run([sys.executable, str(ROOT / "scripts/benchmark_unified_retrieval.py"), "run"], cwd=ROOT, check=True)

policy = read(RUN / "locked_policy.json")
assert policy == read(ROOT / "config/unified_retrieval.json"), "Runtime policy differs from Dev lock"
with connect() as conn:
    active = conn.execute("SELECT snapshot,profile FROM rag.active").fetchone()
assert active == {"snapshot": policy["snapshot"], "profile": policy["profile"]}
display(policy)


## 2. Kết quả Test thực tế

HS đo Top-10 HS8 chiếu từ cùng pool đã chấm; `mixed_*` đo vị trí trong danh sách nguồn hỗn hợp. Legal đòi mọi nhóm ID nguồn sơ cấp trong Top-10, gói nguồn bổ sung được báo riêng. Baseline v4 gọi đúng tác vụ đã biết; v5 dùng Auto chung. Không dùng Test để chọn lại cấu hình.


In [ ]:
result = read(RUN / "test_metrics.json")
assert result["policy_hash"] == digest(policy)
display({k: v for k, v in result.items() if k != "rows"})
display(read(RUN / "test_comparison.json")["intervals"])
from IPython.display import Markdown
display(Markdown((RUN / "IMPLEMENTATION_REPORT_V5.md").read_text()))


## 3. Demo chung cho mọi câu hỏi

`matches`: Top-10 đoạn nguồn hỗn hợp. `hs_candidates`: shortlist HS8 từ cùng ranking. `package`: nguồn sơ cấp thực sự được trích trong context. `semantic_intent` không ảnh hưởng kết quả tìm kiếm. Không có kết luận HS tự động. Demo tắt mặc định vì cần nạp mô hình.


In [ ]:
if RUN_RUNTIME_DEMO:
    from rag.embedding import Embedder
    from rag.retrieval import Retriever
    with connect() as conn:
        retriever = Retriever(conn, Embedder())
        for query in ["Ngọc trai nuôi cấy đã gia công", "Quy tắc phân loại hàng hóa chưa hoàn chỉnh được áp dụng như thế nào?"]:
            out = retriever.routed_search(query)
            display({"query":query, "routing":out["routing"], "semantic_intent":out["semantic_intent"], "sources":[{"id":r["id"],"code":r.get("code"),"locator":r.get("locator")} for r in out["matches"]], "hs_codes":[r["code"] for r in out["hs_candidates"]]})
